#### CHATBOT USING LANGCHAIN - Only It is able to have a conversation and remember previous interactions 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

d:\GENERATIVE_AI_KRISH\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018968AFD340>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018968C240B0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Himanshu and I am a Junior AI Engineer")])

AIMessage(content="Nice to meet you, Himanshu. Congratulations on being a Junior AI Engineer. It's a fascinating field with a lot of opportunities for growth and innovation. \n\nWhat specific areas of AI are you interested in or working on? Are you into deep learning, natural language processing, computer vision, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 50, 'total_tokens': 114, 'completion_time': 0.151257378, 'completion_tokens_details': None, 'prompt_time': 0.003739405, 'prompt_tokens_details': None, 'queue_time': 0.075627565, 'total_time': 0.154996783}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1892-3ec6-72a0-85e5-23f561c61722-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 64, 'total_tokens': 114})

In [5]:
from langchain_core.messages import HumanMessage, AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Himanshu and I am a Junior AI Engineer"),
        AIMessage(content="Nice to meet you, Himanshu. Congratulations on being a Junior AI Engineer. That's a fascinating field. What kind of projects or areas are you currently working on or interested in? Natural Language Processing, Computer Vision, or something else?"),
        HumanMessage(content="Hey what is my name and what do I do ?")
    ]
)

AIMessage(content="Your name is Himanshu, and you're a Junior AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 120, 'total_tokens': 136, 'completion_time': 0.011320133, 'completion_tokens_details': None, 'prompt_time': 0.017759472, 'prompt_tokens_details': None, 'queue_time': 0.047289014, 'total_time': 0.029079605}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1892-44bc-7b90-ae33-5847cf3e85bd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 16, 'total_tokens': 136})

##### Message History

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
# User 1 
config={"configurable" : {"session_id" : "user1"}}

response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Himanshu and I am a Junior AI Engineer")],
    config=config
)
response.content

"Nice to meet you, Himanshu. As a Junior AI Engineer, you're likely working on exciting projects that involve machine learning, deep learning, and natural language processing. What specific areas of AI are you interested in or have you been working on recently?"

In [8]:
config={"configurable" : {"session_id" : "user1"}}

response=with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config
)
response.content

'Your name is Himanshu.'

In [9]:
#User 2
config={"configurable" : {"session_id" : "user2"}}

response=with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't retain any context from previous conversations.\n\nIf you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation."

##### USING PROMPT TEMPLATES

In [10]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system", "You are a helpful assistant. Answer all questions to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [11]:
with_message_history=RunnableWithMessageHistory(
    chain, 
    get_session_history, 
    input_messages_key="messages"
)

In [12]:
config={"configurable" : {"session_id" : "Chat1"}}

with_message_history.invoke(
    {"messages" : [HumanMessage(content="Hi, my name is Himanshu")], "language" : "hindi"},
    config=config
)

AIMessage(content='नमस्ते Himanshu जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न या समस्या है जिसका मैं समाधान कर सकता हूँ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 60, 'total_tokens': 122, 'completion_time': 0.08496767, 'completion_tokens_details': None, 'prompt_time': 0.003390988, 'prompt_tokens_details': None, 'queue_time': 0.19935596, 'total_time': 0.088358658}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1892-588d-7243-9168-6d5483278b97-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 62, 'total_tokens': 122})

In [13]:
config={"configurable" : {"session_id" : "Chat1"}}

with_message_history.invoke(
    {"messages" : [HumanMessage(content="What is my name")], "language" : "hindi"},
    config=config
)

AIMessage(content='आपका नाम हिमांशु है।', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 135, 'total_tokens': 148, 'completion_time': 0.012903368, 'completion_tokens_details': None, 'prompt_time': 0.00955672, 'prompt_tokens_details': None, 'queue_time': 0.050800136, 'total_time': 0.022460088}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1892-5b3c-7fd2-8bff-0e5a3563da6b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 13, 'total_tokens': 148})

#### Managing the conversation history

In [14]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer=trim_messages(
    max_tokens=80,
    token_counter=model,
    strategy="last",
    allow_partial=False,
    include_system=True,
    start_on="human"
)

messages=[
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm Himanshu"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like butterscotch ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 3 * 2"),
    AIMessage(content="6"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

d:\GENERATIVE_AI_KRISH\venv\Lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm Himanshu", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like butterscotch ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 3 * 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='6', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additio

In [15]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)|prompt|model
)

response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What ice cream do i like")],
        "language":"English"
    }
)
response.content

'You like butterscotch ice cream!'

In [16]:
response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What math problem did i ask")],
        "language":"English"
    }
)
response.content

'You asked for the answer to the math problem 3 * 2.'

In [17]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [18]:
config={"configurable" : {"session_id" : "chat2"}}

##Wrapping this in the message history
with_message_history.invoke(
    {
        "messages" : [HumanMessage(content="What math problem did i ask")],
        "language" : "English"
    },
    config=config
)


AIMessage(content="You didn't ask a math problem yet. This conversation just started. What would you like to ask or what math problem would you like to solve?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 58, 'total_tokens': 89, 'completion_time': 0.047906241, 'completion_tokens_details': None, 'prompt_time': 0.002818419, 'prompt_tokens_details': None, 'queue_time': 0.15870382, 'total_time': 0.05072466}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1892-87ac-75a2-86fb-4b2aeb4f46c9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 31, 'total_tokens': 89})

##  VECTOR RETRIEVER

In [19]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    )
]

In [20]:
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001894C5D70E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001896A4DBB30>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 226.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
from langchain_chroma import Chroma

vectorStore=Chroma.from_documents(documents,embeddings)
vectorStore

In [23]:
vectorStore.similarity_search("cat")

[Document(id='c1aa58be-a470-4423-bec7-0efb30e7101d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c0404b4e-acd2-429d-869e-a68667300062', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='e622afe3-877d-4f2d-b5b0-ec67ee0e4188', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(id='92e7ac08-a34f-4ed4-835b-434d3c0ac3b7', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [26]:
await vectorStore.asimilarity_search("cat")

[Document(id='c1aa58be-a470-4423-bec7-0efb30e7101d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c0404b4e-acd2-429d-869e-a68667300062', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='e622afe3-877d-4f2d-b5b0-ec67ee0e4188', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(id='92e7ac08-a34f-4ed4-835b-434d3c0ac3b7', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

### USING RETRIVER INSTEAD OF VECTORSTORE 
##### Retriver has runnable but vectorestore has not runnable

In [28]:
## First type of retriever
from typing import List
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorStore.similarity_search).bind(k=1)
retriever.batch(["cat", "Dog"])


[[Document(id='c1aa58be-a470-4423-bec7-0efb30e7101d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='c0404b4e-acd2-429d-869e-a68667300062', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [29]:
## Second type of retriever
retriever=vectorStore.as_retriever(
    search_type="similarity",
    search_kwargs={"k" : 1}
)

retriever.batch(["cat", "Dog"])

[[Document(id='c1aa58be-a470-4423-bec7-0efb30e7101d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='c0404b4e-acd2-429d-869e-a68667300062', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

#### BASIC RAG

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message= """"
Answer this question using the provided context only.
{question}

Context :
{context}
"""

prompt = ChatPromptTemplate([("human", message)])

rag_chain={"context" : retriever, "question":RunnablePassthrough()} |prompt |model

response=rag_chain.invoke("Tell me about cat")
response.content

'According to the provided context, cats are independent pets that often enjoy their own space.'